# Notebook 12 - Hybrid ROI Fallback Ablation

Runs ROI inference when the router primary bbox is valid, otherwise records an explicit full-image fallback for adapter-eligible crop/part results.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

CLONE_TARGET = Path('/content/bitirmeprojesi')
REPO_URL = os.environ.get('AADS_REPO_URL', 'https://github.com/EfeErim/bitirmeprojesi.git')

def _ensure_aads_repo_on_path():
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents, CLONE_TARGET, Path('/content/bitirmeprojesi'), Path('/content/bitirme projesi')]
    for candidate in candidates:
        if (candidate / 'src').is_dir() and (candidate / 'scripts').is_dir():
            repo_root = candidate.resolve()
            if str(repo_root) not in sys.path:
                sys.path.insert(0, str(repo_root))
            os.chdir(repo_root)
            return repo_root
    if not CLONE_TARGET.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(CLONE_TARGET)], check=True)
    if str(CLONE_TARGET) not in sys.path:
        sys.path.insert(0, str(CLONE_TARGET))
    os.chdir(CLONE_TARGET)
    return CLONE_TARGET

ROOT = _ensure_aads_repo_on_path()
from scripts.colab_notebook_bootstrap_helpers import setup_notebook_environment
ROOT = setup_notebook_environment(install_requirements=True, print_tokens=True)
print(f'[BOOT] ROOT={ROOT}')


In [ ]:
from pathlib import Path
from scripts.colab_roi_ablation import ABLATION_CONFIGS, commit_and_push_ablation_results, run_ablation_folder

ABLATION_NAME = 'hybrid_roi_fallback'
IMAGE_DIR = ROOT / 'data' / 'prepared_runtime_datasets' / 'tomato__fruit' / 'test'
ADAPTER_ROOT = ROOT / 'runs' / 'tomato' / 'fruit' / 'tomato_fruit_2026-05-26_10-34-27' / 'outputs' / 'colab_notebook_training'
ADAPTER_CROP = 'tomato'
ADAPTER_PART = 'fruit'
CONFIG_ENV = 'colab'
DEVICE = 'cuda'
LABEL_FROM_PARENT = True
RETURN_OOD = True
OUTPUT_DIR = ROOT / ABLATION_CONFIGS[ABLATION_NAME]['output_dir']

print(f'[CONFIG] ablation={ABLATION_NAME} output={OUTPUT_DIR}')


In [ ]:
if not Path(IMAGE_DIR).is_dir():
    raise FileNotFoundError(f'IMAGE_DIR not found: {IMAGE_DIR}')
if not Path(ADAPTER_ROOT).is_dir():
    raise FileNotFoundError(f'ADAPTER_ROOT not found: {ADAPTER_ROOT}')

report = run_ablation_folder(
    IMAGE_DIR,
    ablation_name=ABLATION_NAME,
    output_dir=OUTPUT_DIR,
    label_from_parent=LABEL_FROM_PARENT,
    adapter_crop=ADAPTER_CROP,
    adapter_part=ADAPTER_PART,
    config_env=CONFIG_ENV,
    device=DEVICE,
    adapter_root=ADAPTER_ROOT,
    return_ood=RETURN_OOD,
)
git_result = commit_and_push_ablation_results(
    OUTPUT_DIR,
    repo_root=ROOT,
    message='Add hybrid ROI fallback ablation results',
)
{'summary': report['summary'], 'git': git_result}
